# RAG Simple: Pipeline 100% Local

## Objetivo

Construir un sistema RAG (Retrieval-Augmented Generation) completamente local que permita hacer preguntas sobre documentos privados usando Llama 3 y embeddings locales.

### ¿Qué vamos a construir?

Un pipeline que:
1. Carga un documento de texto
2. Lo divide en fragmentos (chunks)
3. Convierte los chunks a embeddings
4. Almacena los embeddings en una base de datos vectorial
5. Busca chunks relevantes para una pregunta
6. Genera una respuesta usando el contexto encontrado

**Todo 100% local, sin APIs externas.**


In [ ]:
# Setup inicial
import sys
import os
from pathlib import Path

# Hack para importar desde src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.models import get_local_llm, get_local_embeddings

print("✓ Imports completados")


## Paso 1: Setup de Datos

Creamos un archivo de texto de ejemplo con contenido legal (política de privacidad) para demostrar el sistema RAG.


In [ ]:
# Crear directorio data si no existe
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)

# Crear archivo de política de privacidad dummy
privacy_file = data_dir / "privacy_policy.txt"
privacy_content = """POLÍTICA DE PRIVACIDAD - EMPRESA TECH SOLUTIONS

FECHA DE ENTRADA EN VIGOR: 1 de enero de 2024

1. INFORMACIÓN QUE RECOPILAMOS

Recopilamos información que usted nos proporciona directamente, incluyendo:
- Nombre completo y datos de contacto (correo electrónico, teléfono)
- Información de facturación y métodos de pago
- Datos de registro de cuenta y preferencias de usuario
- Comunicaciones que mantiene con nuestro servicio de atención al cliente

También recopilamos automáticamente información técnica:
- Dirección IP y datos de localización aproximada
- Tipo de navegador y sistema operativo
- Páginas visitadas y tiempo de permanencia
- Cookies y tecnologías de seguimiento similares

2. CÓMO UTILIZAMOS SU INFORMACIÓN

Utilizamos la información recopilada para:
- Proporcionar, mantener y mejorar nuestros servicios
- Procesar transacciones y enviar notificaciones relacionadas
- Personalizar su experiencia y mostrar contenido relevante
- Enviar comunicaciones de marketing (con su consentimiento)
- Detectar y prevenir fraudes y actividades no autorizadas
- Cumplir con obligaciones legales y regulatorias

3. COMPARTIR INFORMACIÓN

No vendemos su información personal. Compartimos información únicamente en las siguientes circunstancias:
- Con proveedores de servicios que nos ayudan a operar (bajo acuerdos de confidencialidad)
- Cuando sea requerido por ley o proceso legal
- Para proteger nuestros derechos, propiedad o seguridad
- En caso de fusión, adquisición o venta de activos (con notificación previa)

4. SUS DERECHOS

Usted tiene derecho a:
- Acceder a la información personal que tenemos sobre usted
- Solicitar corrección de datos inexactos o incompletos
- Solicitar eliminación de su información personal
- Oponerse al procesamiento de su información
- Solicitar portabilidad de sus datos
- Retirar su consentimiento en cualquier momento

Para ejercer estos derechos, contacte a: privacidad@techsolutions.com

5. SEGURIDAD DE DATOS

Implementamos medidas de seguridad técnicas y organizativas:
- Cifrado de datos en tránsito (TLS/SSL)
- Cifrado de datos en reposo
- Control de acceso basado en roles
- Auditorías de seguridad regulares
- Capacitación del personal en protección de datos

6. RETENCIÓN DE DATOS

Conservamos su información personal durante el tiempo necesario para:
- Cumplir con los propósitos para los que fue recopilada
- Cumplir con obligaciones legales, contables o de informes
- Resolver disputas y hacer cumplir nuestros acuerdos

Los datos se eliminan de forma segura cuando ya no son necesarios.

7. COOKIES Y TECNOLOGÍAS DE SEGUIMIENTO

Utilizamos cookies esenciales, de rendimiento y de funcionalidad.
Puede gestionar sus preferencias de cookies a través de la configuración de su navegador.
Las cookies de marketing requieren su consentimiento explícito.

8. CAMBIOS A ESTA POLÍTICA

Nos reservamos el derecho de actualizar esta política de privacidad.
Le notificaremos cambios significativos por correo electrónico o mediante aviso en nuestro sitio web.
La fecha de "última actualización" se modificará en consecuencia.

9. CONTACTO

Para preguntas sobre esta política de privacidad:
Email: privacidad@techsolutions.com
Teléfono: +1-800-555-0123
Dirección: 123 Tech Street, San Francisco, CA 94105
"""

# Escribir el archivo
with open(privacy_file, "w", encoding="utf-8") as f:
    f.write(privacy_content)

print(f"✓ Archivo creado: {privacy_file}")
print(f"  Tamaño: {len(privacy_content)} caracteres")


In [ ]:
# Cargar el documento usando TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader(str(privacy_file), encoding="utf-8")
documents = loader.load()

print(f"✓ Documento cargado")
print(f"  Número de documentos: {len(documents)}")
print(f"  Tamaño del contenido: {len(documents[0].page_content)} caracteres")
print(f"\nPrimeros 200 caracteres:")
print("-" * 60)
print(documents[0].page_content[:200] + "...")


## Paso 2: Splitting (Troceado)

### ¿Por qué dividir el texto en chunks?

1. **Límites de contexto**: Los modelos LLM tienen límites en el tamaño del contexto que pueden procesar
2. **Búsqueda precisa**: Fragmentos pequeños permiten encontrar información específica más fácilmente
3. **Eficiencia**: Es más eficiente buscar en chunks pequeños que en documentos completos
4. **Relevancia**: Podemos recuperar solo los fragmentos relevantes para una pregunta específica

Visualicemos cómo se divide el texto:


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Crear el splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # Tamaño máximo de cada chunk (caracteres)
    chunk_overlap=50,   # Solapamiento entre chunks (mantiene contexto)
    length_function=len,
)

# Dividir el documento
chunks = text_splitter.split_documents(documents)

print(f"✓ Documento dividido en {len(chunks)} chunks")
print(f"\nDistribución de tamaños:")
for i, chunk in enumerate(chunks[:5], 1):  # Mostrar primeros 5
    print(f"  Chunk {i}: {len(chunk.page_content)} caracteres")

print(f"\nEjemplo de chunk (Chunk 1):")
print("=" * 60)
print(chunks[0].page_content)
print("=" * 60)


### Visualización del Solapamiento

El `chunk_overlap=50` significa que cada chunk comparte 50 caracteres con el anterior. Esto ayuda a mantener el contexto cuando una idea se divide entre dos chunks.


In [ ]:
# Mostrar el solapamiento entre chunks
if len(chunks) >= 2:
    chunk1_end = chunks[0].page_content[-100:]
    chunk2_start = chunks[1].page_content[:100]
    
    print("Final del Chunk 1:")
    print(chunk1_end)
    print("\n" + "-" * 60)
    print("Inicio del Chunk 2:")
    print(chunk2_start)
    print("\n✓ Los últimos caracteres del Chunk 1 se solapan con los primeros del Chunk 2")


## Paso 3: Embeddings & Vector Store (Local)

### Embeddings Locales

Los embeddings convierten texto en vectores numéricos que capturan el significado semántico. Textos similares tienen vectores similares.

### Vector Store Persistente

Usaremos **ChromaDB** como base de datos vectorial local. Al usar `persist_directory`, los datos se guardan en disco y pueden reutilizarse entre sesiones.


In [ ]:
from langchain_community.vectorstores import Chroma

# Obtener embeddings locales
embeddings = get_local_embeddings()

print(f"✓ Embeddings configurados: {embeddings.model_name}")

# Crear vector store persistente
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Persistir en disco
)

print(f"✓ Vector Store creado con {len(chunks)} documentos")
print(f"  Ubicación: ./chroma_db")
print(f"  Los datos se guardarán en disco para reutilización")


## Paso 4: Retrieval (Búsqueda)

El **retriever** busca los chunks más relevantes para una pregunta. Convierte la pregunta en un embedding y encuentra los chunks con embeddings más similares.


In [ ]:
# Crear el retriever desde el vectorstore
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # Devolver los 3 chunks más relevantes
)

print("✓ Retriever configurado (top 3 resultados)")

# Probar el retriever con una pregunta
pregunta_test = "¿Qué información personal recopilan?"
docs_recuperados = retriever.invoke(pregunta_test)

print(f"\n✓ Búsqueda realizada para: '{pregunta_test}'")
print(f"  Documentos recuperados: {len(docs_recuperados)}")
print("\n" + "=" * 60)
print("CHUNKS RECUPERADOS:")
print("=" * 60)

for i, doc in enumerate(docs_recuperados, 1):
    print(f"\n--- Chunk {i} ({len(doc.page_content)} caracteres) ---")
    print(doc.page_content[:400] + "..." if len(doc.page_content) > 400 else doc.page_content)


## Paso 5: Generación (The RAG Chain)

Ahora construimos la cadena RAG completa usando LCEL. La cadena combina:

1. **Retriever**: Busca chunks relevantes
2. **Prompt**: Formatea pregunta + contexto
3. **LLM**: Genera la respuesta
4. **Parser**: Extrae el texto limpio

### El Prompt RAG

El prompt estándar de RAG incluye:
- **Contexto**: Los chunks recuperados
- **Pregunta**: La pregunta del usuario
- **Instrucciones**: Cómo usar el contexto para responder


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Definir el prompt RAG estándar
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente experto que responde preguntas basándote ÚNICAMENTE en el contexto proporcionado.
    
Si la respuesta no está en el contexto, di claramente que no tienes esa información.
Responde de forma clara, concisa y profesional."""),
    ("human", """Contexto:
{context}

Pregunta: {question}

Respuesta:""")
])

print("✓ Prompt RAG definido")


### Construir la Cadena LCEL

La cadena RAG se construye usando el operador pipe (`|`) de LCEL:

```python
rag_chain = (
    {
        "context": retriever,                    # Busca chunks relevantes
        "question": RunnablePassthrough()        # Pasa la pregunta tal cual
    }
    | prompt                                      # Formatea prompt
    | llm                                         # Genera respuesta
    | StrOutputParser()                          # Extrae texto
)
```

### ¿Qué es RunnablePassthrough?

`RunnablePassthrough()` es un componente especial que **pasa el input sin modificarlo**. En este caso, toma la pregunta del usuario y la pasa directamente al prompt sin procesarla.

En el diccionario:
- `"context": retriever` → El retriever busca chunks y los convierte en texto
- `"question": RunnablePassthrough()` → La pregunta se pasa tal cual

Esto crea un diccionario `{"context": "...", "question": "..."}` que el prompt usa para formatear el mensaje final.


In [ ]:
# Instanciar el modelo local
llm = get_local_llm()

# Función para formatear los documentos recuperados
def format_docs(docs):
    """Convierte una lista de documentos en un string formateado."""
    return "\n\n".join(doc.page_content for doc in docs)

# Construir la cadena RAG completa usando LCEL
rag_chain = (
    {
        "context": retriever | format_docs,      # Retriever → formatear docs → context
        "question": RunnablePassthrough()        # Pasar pregunta tal cual
    }
    | prompt                                      # Formatear prompt
    | llm                                         # Generar respuesta
    | StrOutputParser()                          # Extraer texto limpio
)

print("✓ Cadena RAG construida")
print("\nFlujo:")
print("  Input (pregunta)")
print("    ↓")
print("  Retriever → format_docs → context")
print("  question (passthrough)")
print("    ↓")
print("  Prompt Template")
print("    ↓")
print("  LLM (Llama 3)")
print("    ↓")
print("  StrOutputParser")
print("    ↓")
print("  Output (respuesta)")


## Paso 6: Ejecución

Ahora podemos hacer preguntas sobre la política de privacidad y obtener respuestas generadas por Llama 3 basadas en el documento.


In [ ]:
# Pregunta sobre la política de privacidad
pregunta = "¿Qué derechos tienen los usuarios sobre sus datos personales?"

print("=" * 60)
print("PREGUNTA:")
print("=" * 60)
print(pregunta)
print("\n" + "=" * 60)
print("RESPUESTA GENERADA POR LLAMA 3:")
print("=" * 60)

respuesta = rag_chain.invoke(pregunta)
print(respuesta)


In [ ]:
# Otra pregunta
pregunta2 = "¿Cómo se protegen los datos personales?"

print("=" * 60)
print("PREGUNTA:")
print("=" * 60)
print(pregunta2)
print("\n" + "=" * 60)
print("RESPUESTA GENERADA POR LLAMA 3:")
print("=" * 60)

respuesta2 = rag_chain.invoke(pregunta2)
print(respuesta2)


In [ ]:
# Pregunta sobre algo que NO está en el documento
pregunta3 = "¿Cuál es la política de reembolsos?"

print("=" * 60)
print("PREGUNTA:")
print("=" * 60)
print(pregunta3)
print("\n" + "=" * 60)
print("RESPUESTA GENERADA POR LLAMA 3:")
print("=" * 60)

respuesta3 = rag_chain.invoke(pregunta3)
print(respuesta3)


## Resumen del Pipeline RAG

Hemos construido un sistema RAG completo con los siguientes componentes:

1. ✅ **Carga de documentos**: `TextLoader` carga el archivo de texto
2. ✅ **División en chunks**: `RecursiveCharacterTextSplitter` divide el texto
3. ✅ **Embeddings locales**: `HuggingFaceEmbeddings` convierte texto a vectores
4. ✅ **Vector Store persistente**: `Chroma` almacena embeddings en disco
5. ✅ **Retriever**: Busca chunks relevantes para preguntas
6. ✅ **Cadena RAG con LCEL**: Combina retriever, prompt, LLM y parser
7. ✅ **Generación local**: Llama 3 genera respuestas basadas en el contexto

### Ventajas de este enfoque:

- 🔒 **100% Local**: Todo se procesa en tu máquina
- 💾 **Persistente**: El vector store se guarda en disco
- 🚀 **Rápido**: Sin latencia de red
- 🔐 **Privado**: Tus datos nunca salen de tu computadora
- 🎯 **Preciso**: Respuestas basadas en tus documentos específicos

### Próximos pasos:

- Cargar múltiples documentos
- Agregar metadatos a los chunks
- Implementar filtrado por metadatos
- Usar streaming para respuestas en tiempo real
- Agregar memoria de conversación


## Notas Técnicas

### Reutilizar el Vector Store

Si ya tienes un vector store creado, puedes cargarlo directamente:

```python
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)
```

### Ajustar el Número de Chunks Recuperados

Puedes cambiar `k` en el retriever para obtener más o menos contexto:

```python
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})  # Top 5
```

### Limpiar el Vector Store

Si necesitas empezar de nuevo:

```python
import shutil
shutil.rmtree("./chroma_db")
```
